## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-14-20-48-54 +0000,nyt,Here’s Why It Could Take Days for Flights to R...,https://www.nytimes.com/2026/11/14/us/faa-grou...
1,2026-11-14-20-48-27 +0000,nypost,Rock legends turn Southwest flight from LAX in...,https://nypost.com/2026/11/14/us-news/rock-ban...
2,2026-11-14-20-47-37 +0000,startribune,Live: Minneota beats Belgrade-Brooten-Elrosa f...,https://www.startribune.com/minnesota-high-sch...
3,2026-11-14-20-47-02 +0000,bbc,UK ethics watchdog chief steps back from role ...,https://www.bbc.co.uk/news/articles/cmewv2kzq2...
4,2026-11-14-20-44-43 +0000,nyt,"Bomb Found at Ethanol Plant in Peoria, Ill., M...",https://www.nytimes.com/2026/11/14/us/bomb-eth...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2421/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
80,trump,25
15,ground,24
59,house,15
242,outage,14
472,nvidia,14


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
205,2026-11-14-00-49-44 +0000,latimes,Democrats press Trump administration for answe...,https://www.latimes.com/politics/story/2026-11...,93
185,2026-11-14-02-07-00 +0000,wsj,"More than almost any CEO, Jensen Huang helped ...",https://www.wsj.com/politics/policy/nvidia-jen...,91
81,2026-11-14-15-53-55 +0000,bbc,"Trump, Putin and Xi plan historic joint meetin...",https://www.bbc.co.uk/news/articles/ckz74jzpy1...,91
190,2026-11-14-02-00-00 +0000,wsj,The White House ordered officials this summer ...,https://www.wsj.com/politics/policy/china-vape...,86
280,2026-11-13-21-06-14 +0000,nypost,"Trump will have historic meeting with Xi, Puti...",https://nypost.com/2026/11/13/us-news/trump-wi...,78


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
205,93,2026-11-14-00-49-44 +0000,latimes,Democrats press Trump administration for answe...,https://www.latimes.com/politics/story/2026-11...
66,71,2026-11-14-17-27-50 +0000,nypost,Musk wants disgraced ex-Nvidia boss Jensen Hua...,https://nypost.com/2026/11/14/world-news/elon-...
81,66,2026-11-14-15-53-55 +0000,bbc,"Trump, Putin and Xi plan historic joint meetin...",https://www.bbc.co.uk/news/articles/ckz74jzpy1...
30,58,2026-11-14-19-39-31 +0000,chitri,New Orleans port sues to get control of tanker...,https://www.chicagotribune.com/2026/11/14/new-...
187,52,2026-11-14-02-03-00 +0000,wsj,California voters had a chance to impose the n...,https://www.wsj.com/politics/california-voters...
157,50,2026-11-14-06-42-27 +0000,bbc,Microsoft says services fully restored after d...,https://www.bbc.co.uk/news/articles/c6rrg51rkz...
286,45,2026-11-13-21-01-00 +0000,wsj,Of all the ways Chairman Kevin Warsh wants to ...,https://www.wsj.com/economy/central-banking/fe...
160,40,2026-11-14-05-50-15 +0000,nypost,Dems launch sexual harassment probe against ne...,https://nypost.com/2026/11/14/us-news/dems-lau...
218,37,2026-11-13-23-52-22 +0000,latimes,"LAX, other airports filled with stranded passe...",https://www.latimes.com/california/story/2026-...
185,34,2026-11-14-02-07-00 +0000,wsj,"More than almost any CEO, Jensen Huang helped ...",https://www.wsj.com/politics/policy/nvidia-jen...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
